In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import pygris

from pathlib import Path


In [3]:
# makes sure it works when notebook runs from notebooks/ or project root
ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data"
PROCESSED = DATA / "processed"

seg = pd.read_csv(
    PROCESSED / "sf_segments_clean.csv",
    usecols=[
        "service_date", "route_id", "direction_id",
        "from_stop_id", "to_stop_id", "period", "delay_s"
    ],
    dtype={
        "route_id": "string",
        "direction_id": "string",
        "from_stop_id": "string",
        "to_stop_id": "string"
    }
)

acs = pd.read_csv(
    PROCESSED / "sf_census_tracts.csv",
    dtype={"GEOID": "string"}
)

catch = pd.read_csv(
    PROCESSED / "sf_stop_catchments.csv",
    dtype={"stop_id": "string"}
)

print("Muni:", seg.shape)
print("Census:", acs.shape)
print("Catchments:", catch.shape)

Muni: (4820690, 7)
Census: (244, 13)
Catchments: (3234, 5)


In [7]:
# want seg DF rows to represent one route segment during specific time period.
seg["is_slower"] = seg["delay_s"] > 0

# calculate number of observed trips through the segment, number of distinct days represented, median delay (typical travel time), mean delay (average excess travel time), and fraction of observations slower than scheduled
summary = (
    seg.groupby([
        "route_id",
        "direction_id",
        "from_stop_id",
        "to_stop_id",
        "period"
    ])
    .agg(
        observations=("delay_s", "size"),
        service_days=("service_date", "nunique"),
        median_delay=("delay_s", "median"),
        mean_delay=("delay_s", "mean"),
        pct_slower=("is_slower", "mean")
    )
    .reset_index()
)
# standardize to minutes and percentage
summary["median_delay"] = summary["median_delay"] / 60
summary["mean_delay"] = summary["mean_delay"] / 60
summary["pct_slower"] = summary["pct_slower"] * 100
summary.head()

,route_id,direction_id,from_stop_id,to_stop_id,period,observations,service_days,median_delay,mean_delay,pct_slower
0,SF:1,0.0,13547,13549,AM peak,346,21,0.000000,0.046291,45.953757
1,SF:1,0.0,13547,13549,PM peak,383,16,-0.066667,-0.044430,29.765013
2,SF:1,0.0,13547,13549,evening,285,16,-0.133333,-0.091637,16.140351
3,SF:1,0.0,13547,13549,midday,586,16,-0.016667,0.126735,39.761092
4,SF:1,0.0,13549,13550,AM peak,346,21,0.183333,0.297158,84.682081


In [8]:
# filter segments with very few observations as stated in proposal
# TODO: consider other thresholds later!
summary = summary[
    (summary["observations"] >= 30)
    & (summary["service_days"] >= 5)
].copy()

print(summary.shape)

(17182, 10)


In [9]:
# combine muni data with census catchments
combined = summary.merge(
    catch,
    left_on="from_stop_id",
    right_on="stop_id",
    how="left",
    validate="many_to_one"
)

combined.head()

,route_id,direction_id,from_stop_id,to_stop_id,period,observations,service_days,median_delay,mean_delay,pct_slower,stop_id,households_in_buffer,pct_without_vehicle,pct_transit_commuters,avg_median_income
0,SF:1,0.0,13547,13549,AM peak,346,21,0.000000,0.046291,45.953757,13547,1484.738706,21.434685,15.753785,129786.295452
1,SF:1,0.0,13547,13549,PM peak,383,16,-0.066667,-0.044430,29.765013,13547,1484.738706,21.434685,15.753785,129786.295452
2,SF:1,0.0,13547,13549,evening,285,16,-0.133333,-0.091637,16.140351,13547,1484.738706,21.434685,15.753785,129786.295452
3,SF:1,0.0,13547,13549,midday,586,16,-0.016667,0.126735,39.761092,13547,1484.738706,21.434685,15.753785,129786.295452
4,SF:1,0.0,13549,13550,AM peak,346,21,0.183333,0.297158,84.682081,13549,1966.780734,20.627790,16.948268,123212.721671


In [10]:
print("Before merge:", summary.shape)
print("After merge:", combined.shape)

Before merge: (17182, 10)
After merge: (17182, 15)


In [11]:
combined[[
    "households_in_buffer",
    "pct_without_vehicle",
    "pct_transit_commuters",
    "avg_median_income"
]].isna().sum()

households_in_buffer      0
pct_without_vehicle       8
pct_transit_commuters     8
avg_median_income        40
dtype: int64

In [12]:
combined[[
    "route_id",
    "from_stop_id",
    "to_stop_id",
    "period",
    "median_delay",
    "observations",
    "pct_without_vehicle",
    "pct_transit_commuters",
    "avg_median_income"
]].head(10)

,route_id,from_stop_id,to_stop_id,period,median_delay,observations,pct_without_vehicle,pct_transit_commuters,avg_median_income
0,SF:1,13547,13549,AM peak,0.000000,346,21.434685,15.753785,129786.295452
1,SF:1,13547,13549,PM peak,-0.066667,383,21.434685,15.753785,129786.295452
2,SF:1,13547,13549,evening,-0.133333,285,21.434685,15.753785,129786.295452
3,SF:1,13547,13549,midday,-0.016667,586,21.434685,15.753785,129786.295452
4,SF:1,13549,13550,AM peak,0.183333,346,20.627790,16.948268,123212.721671
5,SF:1,13549,13550,PM peak,0.150000,400,20.627790,16.948268,123212.721671
6,SF:1,13549,13550,evening,0.100000,285,20.627790,16.948268,123212.721671
7,SF:1,13549,13550,midday,0.200000,574,20.627790,16.948268,123212.721671
8,SF:1,13550,14277,AM peak,1.000000,341,19.089998,16.952067,126883.541420
9,SF:1,13550,14277,PM peak,1.083333,396,19.089998,16.952067,126883.541420


In [13]:
# stop_id is redundant as they go in loops
combined = combined.drop(columns=["stop_id"])

In [16]:
combined.sort_values(
    "median_delay",
    ascending=False
)[[
    "route_id",
    "from_stop_id",
    "to_stop_id",
    "period",
    "median_delay",
    "observations",
    "pct_without_vehicle",
    "avg_median_income"
]].head(20)

,route_id,from_stop_id,to_stop_id,period,median_delay,observations,pct_without_vehicle,avg_median_income
11084,SF:54,16485,17606,midday,19.658333,282,7.918893,141011.787472
11081,SF:54,16485,17606,AM peak,14.883333,116,7.918893,141011.787472
11082,SF:54,16485,17606,PM peak,11.966667,192,7.918893,141011.787472
15100,SF:F,15686,13311,PM peak,10.783333,287,37.904090,201869.643380
16356,SF:N,15211,15213,evening,9.716667,253,12.028041,136474.023961
4578,SF:28,15467,17038,midday,9.566667,459,35.948985,109808.582641
4575,SF:28,15467,17038,AM peak,9.250000,215,35.948985,109808.582641
4577,SF:28,15467,17038,evening,9.200000,233,35.948985,109808.582641
4576,SF:28,15467,17038,PM peak,9.133333,306,35.948985,109808.582641
15169,SF:F,15175,15184,midday,9.016667,463,35.257696,100744.190882


In [24]:
combined.to_csv(
    PROCESSED / "sf_muni_census_combined.csv",
    index=False
)